# 02 — BCRD CPI & Family Basket | Silver Transformation

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Parse the six BCRD presentation workbooks captured in Bronze into **tidy, typed, validated monthly tables** that support the inflation and cost-of-living KPIs.

## Lineage
Source: `bronze.bcrd_cpi_raw_rows` → parsers in [`src/shared/bcrd_cpi_parsers.py`](../shared/bcrd_cpi_parsers.py)

| Target table | Grain | Coverage |
|---|---|---|
| `silver.bcrd_cpi_national_monthly` | Month | 1984 onward |
| `silver.bcrd_cpi_core_monthly` | Month | 2000 onward |
| `silver.bcrd_cpi_groups_monthly` | Month × expenditure group (12) | 1999 onward |
| `silver.bcrd_cpi_regions_monthly` | Month × region (4) | 2011 onward |
| `silver.bcrd_family_basket_monthly` | Month × segment (5 quintiles, national, 4 regions) | 2018 onward |

## Rules
- Spanish month names and the "year written once per block" layout are resolved into a real `reference_month` (first day of the month).
- `-` in the source means "not applicable" and becomes NULL; any other non-numeric value is **rejected** to `quality.dq_rejected_records`.
- Region labels are standardized (e.g. "Región Norte" and "Región Norte o Cibao" → `Norte o Cibao`); the source label is kept.
- Family-basket quintile values before Oct-2020 are a BCRD **retropolation** (methodology note) and are flagged `is_retropolated`.
- Published variations are kept as published. Silver also checks that each published monthly variation matches the one implied by the index.

**Load pattern:** full refresh from Bronze. **Status:** Development — not production-ready.

In [0]:
# ============================================================
# Step 1 — Configuration and shared parsers
# ============================================================
import sys, uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Import parsers from src/shared (works from the Git folder and from any copy of src/)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
src_root = "/Workspace" + notebook_path.rsplit("/", 2)[0]
sys.path.insert(0, f"{src_root}/shared")
import importlib, bcrd_cpi_parsers as P
importlib.reload(P)

CATALOG = "rd_economic_intelligence"
SOURCE_TABLE = f"{CATALOG}.bronze.bcrd_cpi_raw_rows"
REJECTED_TABLE = f"{CATALOG}.quality.dq_rejected_records"
RULE_RESULTS_TABLE = f"{CATALOG}.quality.dq_rule_results"
T = {
    "national": f"{CATALOG}.silver.bcrd_cpi_national_monthly",
    "core":     f"{CATALOG}.silver.bcrd_cpi_core_monthly",
    "groups":   f"{CATALOG}.silver.bcrd_cpi_groups_monthly",
    "regions":  f"{CATALOG}.silver.bcrd_cpi_regions_monthly",
    "basket":   f"{CATALOG}.silver.bcrd_family_basket_monthly",
}
BASE_PERIOD = "Oct 2019 - Sep 2020 = 100"
MOM_TOLERANCE_PP = 0.05   # published vs index-implied monthly variation, percentage points

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
print(f"Run ID: {RUN_ID} | parsers: {P.__file__}")

In [0]:
# ============================================================
# Step 2 — Load Bronze rows per dataset (latest batch per file)
# ============================================================
bronze = spark.table(SOURCE_TABLE)
latest = bronze.groupBy("_source_file").agg(F.max("_ingestion_ts").alias("_ingestion_ts"))
bronze = bronze.join(latest, ["_source_file", "_ingestion_ts"]).filter("_source_sheet_index = 0")

lineage = {r["dataset_code"]: (r["_source_file"], r["_batch_id"])
           for r in bronze.select("dataset_code", "_source_file", "_batch_id").distinct().collect()}

def rows_for(code):
    data = (bronze.filter(F.col("dataset_code") == code)
            .select("_source_row_number", "cells").orderBy("_source_row_number").collect())
    return [(r["_source_row_number"], list(r["cells"] or [])) for r in data]

raw = {code: rows_for(code) for code in lineage}
print({k: len(v) for k, v in raw.items()})

In [0]:
# ============================================================
# Step 3 — Parse each layout
# ============================================================
parsed, rejects, dims = {}, [], {}

parsed["national"], rej = P.parse_cpi_national(raw["cpi_national"]); rejects += rej
parsed["core"], rej = P.parse_cpi_core(raw["cpi_core"]); rejects += rej
parsed["groups"], rej, dims["groups"] = P.parse_cpi_groups(raw["cpi_groups"]); rejects += rej
parsed["regions"], rej, dims["regions"] = P.parse_cpi_regions(raw["cpi_regions"]); rejects += rej
bq, rej, dims["basket_quintiles"] = P.parse_basket(raw["basket_quintiles"], "basket_quintiles", "income_quintile"); rejects += rej
br, rej, dims["basket_regions"] = P.parse_basket(raw["basket_regions"], "basket_regions", "region"); rejects += rej
for r in bq: r["dataset"] = "basket_quintiles"
for r in br: r["dataset"] = "basket_regions"
parsed["basket"] = bq + br

for k, v in parsed.items():
    print(f"{k:<9} {len(v):>6,} records")
print(f"rejected  {len(rejects):>6,}")
print("Dimensions:", json.dumps(dims, ensure_ascii=False))

In [0]:
# ============================================================
# Step 4 — Build typed DataFrames with lineage
# ============================================================
def to_df(records, schema, dataset_codes):
    df = spark.createDataFrame(records, schema)
    if isinstance(dataset_codes, str):
        file, batch = lineage[dataset_codes]
        df = df.withColumn("_source_file", F.lit(file)).withColumn("_bronze_batch_id", F.lit(batch))
    else:  # basket: two source files
        m = F.create_map(*[x for c in dataset_codes for x in (F.lit(c), F.lit(lineage[c][0]))])
        b = F.create_map(*[x for c in dataset_codes for x in (F.lit(c), F.lit(lineage[c][1]))])
        df = df.withColumn("_source_file", m[F.col("dataset")]).withColumn("_bronze_batch_id", b[F.col("dataset")])
    return (df.withColumn("base_period", F.lit(BASE_PERIOD))
              .withColumn("_silver_run_id", F.lit(RUN_ID))
              .withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))

def dec(c, p=12, s=4):
    return F.col(c).cast(f"decimal({p},{s})")

national = to_df(parsed["national"],
    "reference_month date, source_row_number int, cpi_index double, mom_pct double, ytd_pct double, yoy_pct double, avg_12m_pct double",
    "cpi_national").select("reference_month", dec("cpi_index").alias("cpi_index"), dec("mom_pct").alias("mom_pct"),
        dec("ytd_pct").alias("ytd_pct"), dec("yoy_pct").alias("yoy_pct"), dec("avg_12m_pct").alias("avg_12m_pct"),
        "base_period", "_source_file", F.col("source_row_number").alias("_source_row_number"), "_bronze_batch_id", "_silver_run_id", "_silver_processed_ts")

core = to_df(parsed["core"],
    "reference_month date, source_row_number int, core_cpi_index double, mom_pct double, ytd_pct double, yoy_pct double, observation_type string",
    "cpi_core").select("reference_month", dec("core_cpi_index").alias("core_cpi_index"), dec("mom_pct").alias("mom_pct"),
        dec("ytd_pct").alias("ytd_pct"), dec("yoy_pct").alias("yoy_pct"), "observation_type",
        "base_period", "_source_file", F.col("source_row_number").alias("_source_row_number"), "_bronze_batch_id", "_silver_run_id", "_silver_processed_ts")

groups = to_df(parsed["groups"],
    "reference_month date, source_row_number int, group_index double, mom_pct double, expenditure_group string",
    "cpi_groups").select("reference_month", "expenditure_group", dec("group_index").alias("group_index"), dec("mom_pct").alias("mom_pct"),
        "base_period", "_source_file", F.col("source_row_number").alias("_source_row_number"), "_bronze_batch_id", "_silver_run_id", "_silver_processed_ts")

regions = to_df(parsed["regions"],
    "reference_month date, source_row_number int, cpi_index double, mom_pct double, region string, region_source_label string",
    "cpi_regions").select("reference_month", "region", "region_source_label", dec("cpi_index").alias("cpi_index"), dec("mom_pct").alias("mom_pct"),
        "base_period", "_source_file", F.col("source_row_number").alias("_source_row_number"), "_bronze_batch_id", "_silver_run_id", "_silver_processed_ts")

basket = to_df(parsed["basket"],
    "reference_month date, source_row_number int, basket_cost_dop double, segment_type string, segment string, segment_source_label string, is_retropolated boolean, dataset string",
    ["basket_quintiles", "basket_regions"]).select("reference_month", "segment_type", "segment", "segment_source_label",
        dec("basket_cost_dop", 14, 2).alias("basket_cost_dop"), "is_retropolated",
        "base_period", "_source_file", F.col("source_row_number").alias("_source_row_number"), "_bronze_batch_id", "_silver_run_id", "_silver_processed_ts")

frames = {"national": national, "core": core, "groups": groups, "regions": regions, "basket": basket}
keys = {"national": ["reference_month"], "core": ["reference_month"], "groups": ["reference_month", "expenditure_group"],
        "regions": ["reference_month", "region"], "basket": ["reference_month", "segment_type", "segment"]}
value_col = {"national": "cpi_index", "core": "core_cpi_index", "groups": "group_index", "regions": "cpi_index", "basket": "basket_cost_dop"}
series_cols = {"national": [], "core": [], "groups": ["expenditure_group"], "regions": ["region"], "basket": ["segment_type", "segment"]}

In [0]:
# ============================================================
# Step 5 — Data-quality rules
# ============================================================
checks = []
def add(table, name, rtype, tested, failed):
    status = "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN")
    checks.append((RUN_ID, table, name, rtype, int(tested), int(failed), status, RUN_TS))

for name, df in frames.items():
    table, n = T[name], df.count()
    add(table, "unique_business_key", "enforced", n, n - df.select(*keys[name]).distinct().count())
    add(table, "positive_value", "enforced", n, df.filter(F.col(value_col[name]).isNull() | (F.col(value_col[name]) <= 0)).count())

    # Contiguous monthly series (no missing months inside each series)
    w = Window.partitionBy(*series_cols[name]).orderBy("reference_month") if series_cols[name] else Window.orderBy("reference_month")
    monthly_only = df.filter(F.col("observation_type") == "monthly") if name == "core" else df
    gaps = (monthly_only.withColumn("_prev", F.lag("reference_month").over(w))
            .filter(F.col("_prev").isNotNull() & (F.months_between("reference_month", "_prev") != 1)).count())
    add(table, "contiguous_months", "enforced", n, gaps)

    # Published monthly variation vs variation implied by the index (observational)
    if "mom_pct" in df.columns:
        implied = (df.withColumn("_prev_idx", F.lag(value_col[name]).over(w))
                     .withColumn("_prev_month", F.lag("reference_month").over(w))
                     .filter(F.col("mom_pct").isNotNull() & (F.months_between("reference_month", "_prev_month") == 1))
                     .withColumn("_implied", (F.col(value_col[name]) / F.col("_prev_idx") - 1) * 100))
        tested = implied.count()
        failed = implied.filter(F.abs(F.col("_implied") - F.col("mom_pct")) > MOM_TOLERANCE_PP).count()
        add(table, "published_mom_matches_index", "observational", tested, failed)

add(T["national"], "parse_rejects", "enforced", sum(len(v) for v in parsed.values()) + len(rejects), len(rejects))

for c in checks:
    print(f"  {c[6]:<4} {c[1].split('.')[-1]:<32} {c[2]:<30} tested={c[4]:>6,} failed={c[5]:,}")

In [0]:
# ============================================================
# Step 6 — Write Silver tables, rejects and rule results
# ============================================================
failed_enforced = [(c[1], c[2]) for c in checks if c[6] == "FAIL"]
assert not failed_enforced, f"Enforced checks failed, nothing written: {failed_enforced}"

for name, df in frames.items():
    df.orderBy(*keys[name]).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T[name])

if rejects:
    spark.createDataFrame(
        [(RUN_ID, "silver.bcrd_cpi_*", r["rule_name"], f"{r['dataset']}:row{r['source_row_number']}",
          json.dumps(r, default=str), lineage.get(r["dataset"], (None, None))[1], RUN_TS) for r in rejects],
        "run_id string, target_table string, rule_name string, record_key string, record_json string, source_batch_id string, detected_ts timestamp",
    ).write.mode("append").saveAsTable(REJECTED_TABLE)

spark.createDataFrame(checks,
    "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp",
).write.mode("append").saveAsTable(RULE_RESULTS_TABLE)

In [0]:
# ============================================================
# Step 7 — Documentation, constraints and verification
# ============================================================
docs = {
    "national": "Silver: BCRD national CPI (base Oct 2019 - Sep 2020 = 100), one row per month, with published monthly, year-to-date (vs December), 12-month and 12-month-average variations.",
    "core":     "Silver: BCRD core CPI (excludes volatile agricultural goods, alcohol, tobacco, fuels, administered and transport services), one row per month.",
    "groups":   "Silver: BCRD CPI by expenditure group (12 groups), one row per month and group, with published monthly variation.",
    "regions":  "Silver: BCRD CPI by region (Ozama, Norte o Cibao, Este, Sur), one row per month and region, with published monthly variation. Ozama = Distrito Nacional + Santo Domingo province.",
    "basket":   "Silver: BCRD family basket cost in DOP by income quintile, national and region, one row per month and segment. Quintile values before Oct-2020 are a BCRD retropolation (is_retropolated).",
}
for name, comment in docs.items():
    spark.sql(f"COMMENT ON TABLE {T[name]} IS '{comment}'")
    spark.sql(f"ALTER TABLE {T[name]} ALTER COLUMN reference_month SET NOT NULL")
    spark.sql(f"ALTER TABLE {T[name]} DROP CONSTRAINT IF EXISTS chk_positive_value")
    spark.sql(f"ALTER TABLE {T[name]} ADD CONSTRAINT chk_positive_value CHECK ({value_col[name]} > 0)")

summary = {}
for name in frames:
    s = spark.sql(f"SELECT COUNT(*) n, MIN(reference_month) first, MAX(reference_month) last FROM {T[name]}").first()
    summary[name] = {"rows": s["n"], "first": str(s["first"]), "last": str(s["last"])}
    print(f"{name:<9} {s['n']:>6,} rows  {s['first']} -> {s['last']}")

latest_cpi = spark.sql(f"SELECT reference_month, cpi_index, mom_pct, ytd_pct, yoy_pct FROM {T['national']} ORDER BY reference_month DESC LIMIT 1").first()
display(spark.table(T["national"]).orderBy(F.col("reference_month").desc()).limit(13))

dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "tables": summary,
    "rejected": len(rejects),
    "dimensions": dims,
    "checks": {f"{c[1].split('.')[-1]}.{c[2]}": f"{c[6]} ({c[5]}/{c[4]})" for c in checks},
    "latest_cpi": {k: str(v) for k, v in latest_cpi.asDict().items()},
}, ensure_ascii=False))